In [5]:

import json
import os
import sys
import textwrap
from collections import Counter, defaultdict
from datetime import datetime
from nltk import word_tokenize
import pandas as pd

DEFAULT_RESPONSES = os.path.join(".", "data", "responses.json")
OUTPUT_DIR = os.path.join(".", "analysis_output")

# Minimum reasoning length (words) to flag low-effort responses
MIN_REASONING_LENGTH = 20


workers_to_exclude = ["A6A0IS7L6OUAB","AKERE2HC8Y7H4","AKERE2HC8Y7H3","AKERE2HC8Y7H5","AKERE2HC8Y7H9"]

In [ ]:
def load_responses(path):
    """Load JSONL responses file into a list of dicts."""
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"  ⚠ Skipping malformed line {lineno}: {e}")
    return records


def flatten_records(records):
    """Flatten response records into one row per explanation rating.

    Each submission contains a `ratings` list with one entry per student
    explanation.  This produces one flat row per individual rating.
    """
    rows = []
    for rec in records:
        base = {
            "worker_id":            rec.get("mturk_worker_id"),
            "completion_code":      rec.get("completion_code"),
            "timestamp_utc":        rec.get("timestamp_utc"),
            "problem_index":        rec.get("problem_index"),
            "problem_id":           rec.get("problem_id"),
            "problem_statement":    rec.get("problem_statement", "")[:120],
            "selected_line":        rec.get("selected_line"),
            "num_explanations":     rec.get("num_explanations"),
            "java_experience":      rec.get("java_experience_level"),
        }

        # Demographics
        demo = rec.get("demographics", {})
        base["demo_experience"] = demo.get("experience")
        base["demo_role"] = demo.get("role")
        base["screener_score"] = demo.get("java_screener_score")
        base["screener_returning"] = demo.get("java_screener_returning", False)

        # Each rating is one explanation
        ratings = rec.get("ratings", [])
        for r in ratings:
            row = dict(base)
            row["explanation_index"] = r.get("explanation_index")
            row["explanation_text"] = r.get("explanation_text", "")[:120]
            row["correctness_rating"] = r.get("correctness_rating")
            row["completeness_rating"] = r.get("completeness_rating")
            row["reasoning"] = r.get("reasoning", "")

            # Quality flags
            reasoning = row["reasoning"] or ""
            row["reasoning_length"] = len(reasoning)
            row["flag_short_reasoning"] = len(reasoning.strip()) < MIN_REASONING_LENGTH
            row["flag_copy_paste"] = _is_likely_copy_paste(
                reasoning, r.get("explanation_text", "")
            )
            rows.append(row)

    return rows


def _is_likely_copy_paste(reasoning, explanation_text):
    """Heuristic: check if reasoning is suspiciously similar to the explanation."""
    if not reasoning or not explanation_text:
        return False
    reason_words = set(reasoning.lower().split())
    expl_words = set(explanation_text.lower().split())
    if not reason_words or not expl_words:
        return False
    overlap = len(reason_words & expl_words) / len(reason_words)
    return overlap > 0.6


# ── Analysis functions ───────────────────────────────────────────────────────

def print_header(title):
    width = 70
    print()
    print("═" * width)
    print(f"  {title}")
    print("═" * width)


def analyze_overview(df, records):
    print_header("OVERVIEW")
    n_submissions = len(records)
    print(f"  Total submissions:          {n_submissions}")
    print(f"  Total explanation ratings:  {len(df)}")
    print(f"  Unique workers:             {df['worker_id'].nunique()}")
    print(f"  Unique problems rated:      {df['problem_index'].nunique()}")

    # Explanations per submission
    expl_counts = df.groupby(["worker_id", "problem_index"]).size()
    print(f"  Explanations per submission: "
          f"min={expl_counts.min()}, max={expl_counts.max()}, avg={expl_counts.mean():.1f}")

    # Time range
    if "timestamp_utc" in df.columns and df["timestamp_utc"].notna().any():
        times = pd.to_datetime(df["timestamp_utc"], errors="coerce").dropna()
        if len(times):
            print(f"  Date range:                 "
                  f"{times.min():%Y-%m-%d %H:%M} → {times.max():%Y-%m-%d %H:%M} UTC")


def analyze_ratings(df):
    print_header("RATING DISTRIBUTIONS")
    print(f"  Total ratings: {len(df)}")

    print("\n  Correctness:")
    for val, count in df["correctness_rating"].value_counts().items():
        print(f"    {val}: {count} ({count/len(df)*100:.1f}%)")

    print("\n  Completeness:")
    for val, count in df["completeness_rating"].value_counts().items():
        print(f"    {val}: {count} ({count/len(df)*100:.1f}%)")

    # Cross-tab
    if df["correctness_rating"].notna().any() and df["completeness_rating"].notna().any():
        print("\n  Correctness × Completeness cross-tab:")
        ct = pd.crosstab(df["correctness_rating"], df["completeness_rating"])
        print(textwrap.indent(ct.to_string(), "    "))


def analyze_quality_flags(df):
    print_header("RESPONSE QUALITY FLAGS")

    short = df["flag_short_reasoning"].sum()
    copy = df["flag_copy_paste"].sum()
    total = len(df)

    print(f"  Short reasoning (<{MIN_REASONING_LENGTH} chars): {short}/{total} ({short/total*100:.1f}%)")
    print(f"  Suspected copy-paste reasoning:    {copy}/{total} ({copy/total*100:.1f}%)")

    lengths = df["reasoning_length"]
    print(f"\n  Reasoning length (chars):")
    print(f"    Mean:   {lengths.mean():.0f}")
    print(f"    Median: {lengths.median():.0f}")
    print(f"    Min:    {lengths.min():.0f}")
    print(f"    Max:    {lengths.max():.0f}")

    flagged = df[df["flag_short_reasoning"] | df["flag_copy_paste"]]
    if not flagged.empty:
        print(f"\n  Flagged workers:")
        for wid, group in flagged.groupby("worker_id"):
            flags = []
            if group["flag_short_reasoning"].any():
                flags.append(f"short×{group['flag_short_reasoning'].sum()}")
            if group["flag_copy_paste"].any():
                flags.append(f"copy×{group['flag_copy_paste'].sum()}")
            print(f"    {wid}: {', '.join(flags)}")


def analyze_workers(df):
    print_header("PER-WORKER SUMMARY")

    rows = []
    for wid, group in df.groupby("worker_id"):
        n_submissions = group.groupby("problem_index").ngroups
        row = {
            "worker_id": wid,
            "problems_rated": n_submissions,
            "total_explanations_rated": len(group),
            "pct_correct": (group["correctness_rating"] == "Correct").mean() * 100,
            "pct_complete": (group["completeness_rating"] == "Complete").mean() * 100,
            "avg_reasoning_len": group["reasoning_length"].mean(),
            "flags_short": group["flag_short_reasoning"].sum(),
            "flags_copy": group["flag_copy_paste"].sum(),
            "experience": group["demo_experience"].iloc[0],
            "role": group["demo_role"].iloc[0],
        }
        rows.append(row)

    worker_df = pd.DataFrame(rows).sort_values("total_explanations_rated", ascending=False)

    print(f"  Workers: {len(worker_df)}")
    print(f"  Problems per worker:")
    print(f"    Mean:   {worker_df['problems_rated'].mean():.1f}")
    print(f"    Median: {worker_df['problems_rated'].median():.0f}")
    print(f"    Max:    {worker_df['problems_rated'].max()}")

    display_cols = ["worker_id", "problems_rated", "total_explanations_rated",
                    "pct_correct", "pct_complete", "avg_reasoning_len",
                    "flags_short", "flags_copy"]
    print(f"\n  Worker details:")
    print(textwrap.indent(worker_df[display_cols].to_string(index=False, float_format="%.1f"), "    "))

    return worker_df


def analyze_problems(df):
    print_header("PER-PROBLEM SUMMARY")

    rows = []
    for pidx, group in df.groupby("problem_index"):
        n_workers = group["worker_id"].nunique()
        n_explanations = group["explanation_index"].nunique()
        row = {
            "problem_index": pidx,
            "num_explanations": n_explanations,
            "num_workers": n_workers,
            "total_ratings": len(group),
            "pct_rated_correct": (group["correctness_rating"] == "Correct").mean() * 100,
            "pct_rated_complete": (group["completeness_rating"] == "Complete").mean() * 100,
            "avg_reasoning_len": group["reasoning_length"].mean(),
            "problem_preview": group["problem_statement"].iloc[0][:80],
        }
        rows.append(row)

    prob_df = pd.DataFrame(rows).sort_values("problem_index")

    print(f"  Problems rated: {len(prob_df)}")
    display_cols = ["problem_index", "num_explanations", "num_workers",
                    "total_ratings", "pct_rated_correct", "pct_rated_complete"]
    print(textwrap.indent(prob_df[display_cols].to_string(index=False, float_format="%.1f"), "    "))

    return prob_df


def analyze_explanations(df):
    """Per-explanation breakdown across all workers."""
    print_header("PER-EXPLANATION SUMMARY")

    rows = []
    for (pidx, eidx), group in df.groupby(["problem_index", "explanation_index"]):
        n_raters = group["worker_id"].nunique()
        row = {
            "problem_index": pidx,
            "explanation_index": eidx,
            "num_raters": n_raters,
            "pct_correct": (group["correctness_rating"] == "Correct").mean() * 100,
            "pct_complete": (group["completeness_rating"] == "Complete").mean() * 100,
            "explanation_preview": group["explanation_text"].iloc[0][:80],
        }
        rows.append(row)

    expl_df = pd.DataFrame(rows).sort_values(["problem_index", "explanation_index"])

    print(f"  Unique explanations: {len(expl_df)}")
    display_cols = ["problem_index", "explanation_index", "num_raters",
                    "pct_correct", "pct_complete"]
    print(textwrap.indent(expl_df[display_cols].to_string(index=False, float_format="%.1f"), "    "))

    return expl_df


def compute_inter_rater(df):
    """Compute simple agreement for explanations rated by multiple workers."""
    print_header("INTER-RATER AGREEMENT")

    multi = df.groupby(["problem_index", "explanation_index"]).filter(
        lambda g: g["worker_id"].nunique() >= 2
    )
    if multi.empty:
        print("  Not enough overlapping ratings (need ≥2 workers per explanation).")
        return

    groups = multi.groupby(["problem_index", "explanation_index"])
    agree_correct = 0
    agree_complete = 0
    total_pairs = 0

    for _, group in groups:
        ratings_c = group["correctness_rating"].dropna().tolist()
        ratings_m = group["completeness_rating"].dropna().tolist()
        n = len(ratings_c)
        for i in range(n):
            for j in range(i + 1, n):
                total_pairs += 1
                if ratings_c[i] == ratings_c[j]:
                    agree_correct += 1
                if i < len(ratings_m) and j < len(ratings_m) and ratings_m[i] == ratings_m[j]:
                    agree_complete += 1

    if total_pairs:
        n_expls = groups.ngroups
        print(f"  Explanations with ≥2 raters: {n_expls}")
        print(f"  Total rater pairs:           {total_pairs}")
        print(f"  Correctness agreement:       "
              f"{agree_correct}/{total_pairs} ({agree_correct/total_pairs*100:.1f}%)")
        print(f"  Completeness agreement:      "
              f"{agree_complete}/{total_pairs} ({agree_complete/total_pairs*100:.1f}%)")


# ── Export functions ─────────────────────────────────────────────────────────

def export_csvs(df, worker_df, prob_df, expl_df, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    all_path = os.path.join(output_dir, "all_ratings.csv")
    df.to_csv(all_path, index=False)

    worker_path = os.path.join(output_dir, "worker_quality.csv")
    worker_df.to_csv(worker_path, index=False)

    prob_path = os.path.join(output_dir, "problem_summary.csv")
    prob_df.to_csv(prob_path, index=False)

    expl_path = os.path.join(output_dir, "explanation_summary.csv")
    expl_df.to_csv(expl_path, index=False)

    return all_path, worker_path, prob_path, expl_path



In [ ]:
path = DEFAULT_RESPONSES

if not os.path.exists(path):
    print(f"Error: File not found — {path}")
    print(f"Usage: python {sys.argv[0]} [path/to/responses.json]")
    sys.exit(1)

print(f"Loading responses from: {path}")
records = load_responses(path)
if not records:
    print("No responses found.")
    sys.exit(0)

flat_rows = flatten_records(records)
df = pd.DataFrame(flat_rows)
df = df[~(df.worker_id.isin(workers_to_exclude))]

# ── Run analyses ──
analyze_overview(df, records)
analyze_ratings(df)
analyze_quality_flags(df)
worker_df = analyze_workers(df)
prob_df = analyze_problems(df)
expl_df = analyze_explanations(df)
compute_inter_rater(df)

# ── Export CSVs ──
all_path, worker_path, prob_path, expl_path = export_csvs(
    df, worker_df, prob_df, expl_df, OUTPUT_DIR
)

print_header("EXPORTS")
print(f"  All ratings:           {all_path}")
print(f"  Worker quality:        {worker_path}")
print(f"  Problem summary:       {prob_path}")
print(f"  Explanation summary:   {expl_path}")
print()


Loading responses from: ./data/responses.json

══════════════════════════════════════════════════════════════════════
  OVERVIEW
══════════════════════════════════════════════════════════════════════
  Total responses:        27
  Unique workers:         19
  Unique problems rated:  26
  Task mode breakdown:
      rate: 18 (66.7%)
      pick: 9 (33.3%)
  Date range:             2026-03-08 06:54 → 2026-03-10 12:26 UTC

══════════════════════════════════════════════════════════════════════
  RATE MODE ANALYSIS
══════════════════════════════════════════════════════════════════════
  Responses: 18

  Correctness ratings:
    Correct: 16 (88.9%)
    Incorrect: 2 (11.1%)

  Completeness ratings:
    Incomplete: 13 (72.2%)
    Complete: 5 (27.8%)

  Correctness × Completeness cross-tab:
    completeness_rating  Complete  Incomplete
    correctness_rating                       
    Correct                     5          11
    Incorrect                   0           2

════════════════════════